# 🏯 10张测试 · 万象熔炉 (Anything XL)
## Colab + 万象熔炉 + ControlNet Canny

In [ ]:
# @title 1. 安装依赖
!pip install diffusers transformers accelerate xformers pillow controlnet-aux -q

import torch, os, json, io, time, random
from pathlib import Path
from PIL import Image
from diffusers import StableDiffusionXLControlNetPipeline, ControlNetModel
import numpy as np

print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# @title 2. 挂载 Drive + 检查模型文件
from google.colab import drive
drive.mount('/content/drive')

DRIVE = Path("/content/drive/MyDrive")

# 查找 .safetensors 文件
for f in DRIVE.glob("*.safetensors"):
    print(f"✅ 找到: {f.name} ({f.stat().st_size/1e9:.1f}GB)")
    
if not list(DRIVE.glob("*.safetensors")):
    print("❌ 未找到 .safetensors 文件！")
    print("请下载 AnythingXL_xl.safetensors 上传到 Google Drive 根目录")
    print("下载地址: https://civitai.com/models/9409 (约6.5GB)")

In [ ]:
# @title 3. 加载 10 张测试卡牌
cards = [
    {"card_id":"QH-P-0001-L05","name":"秦始皇","type":"person","dynasty":"秦汉",
     "tags":["帝王","秦朝","统一","长城"],
     "story":"秦始皇嬴政，中国历史上第一位皇帝，统一六国，建立中央集权制度。"},
    {"card_id":"SG-P-0002-L04","name":"关羽","type":"person","dynasty":"三国",
     "tags":["蜀汉","五虎将","忠义","武圣"],
     "story":"关羽，蜀汉五虎将之首，被后世尊为武圣关帝。"},
    {"card_id":"ST-P-0001-L05","name":"李世民","type":"person","dynasty":"隋唐",
     "tags":["唐朝","帝王","贞观之治"],
     "story":"唐太宗李世民，开创贞观之治的一代明君。"},
    {"card_id":"QH-E-0017-L03","name":"鸿门宴","type":"event","dynasty":"秦汉",
     "tags":["楚汉","宴会","刺杀"],
     "story":"项羽设宴鸿门欲杀刘邦，刘邦在张良樊哙帮助下化险为夷。"},
    {"card_id":"SG-E-0018-L04","name":"赤壁之战","type":"event","dynasty":"三国",
     "tags":["三国","火攻","曹操","周瑜"],
     "story":"赤壁之战孙刘联军火烧曹船，奠定三国鼎立格局。"},
    {"card_id":"QH-L-0004-L01","name":"咸阳","type":"place","dynasty":"秦汉",
     "tags":["秦都","关中","宫殿","帝国"],
     "story":"咸阳是秦朝都城，中国历史上第一座大一统帝国首都。"},
    {"card_id":"SG-L-0001-L01","name":"赤壁","type":"place","dynasty":"三国",
     "tags":["长江","火攻","赤壁之战","战场"],
     "story":"赤壁之战火烧连船，三国鼎立格局从此奠定。"},
    {"card_id":"QH-W-0025-L03","name":"天子剑","type":"weapon","dynasty":"秦汉",
     "tags":["帝王","宝剑","象征"],
     "story":"天子剑象征皇权至高无上，秦始皇佩剑为太阿剑。"},
    {"card_id":"QH-B-0029-L03","name":"九章算术","type":"classic","dynasty":"秦汉",
     "tags":["数学","汉代","科学"],
     "story":"九章算术是汉代最终成书的中国古代最重要的数学经典。"},
    {"card_id":"QH-D-0035-L05","name":"大秦帝国","type":"dynasty","dynasty":"秦汉",
     "tags":["秦朝","帝国","统一"],
     "story":"大秦帝国是中国历史上第一个大一统王朝。"},
]

print(f"📋 测试卡牌: {len(cards)} 张")
for c in cards:
    print(f"  {c['card_id']:20s} {c['name']:8s} [{c['type']}]")

In [ ]:
# @title 4. 加载万象熔炉模型
from controlnet_aux import CannyDetector
canny = CannyDetector()

# 找到模型文件
model_files = list(DRIVE.glob("*.safetensors"))
if not model_files:
    raise FileNotFoundError("请先上传 .safetensors 模型文件到 Google Drive!")

model_path = str(model_files[0])
print(f"📦 模型: {model_files[0].name}")

controlnet = ControlNetModel.from_pretrained(
    "diffusers/controlnet-canny-sdxl-1.0", torch_dtype=torch.float16
)

pipe = StableDiffusionXLControlNetPipeline.from_single_file(
    model_path, controlnet=controlnet,
    torch_dtype=torch.float16, use_safetensors=True,
)
pipe = pipe.to("cuda")
pipe.enable_xformers_memory_efficient_attention()

print("✅ 模型加载完成!")
import gc; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# @title 5. 提示词
def make_prompt(card):
    name = card["name"]
    ctype = card["type"]
    dynasty = card.get("dynasty", "")
    tags = card.get("tags", [])
    if isinstance(tags, str):
        try: tags = json.loads(tags)
        except: tags = [t.strip() for t in tags.split(",")]
    tag_str = ", ".join(tags[:4]) if tags else ""
    
    base = "masterpiece, best quality, highly detailed"
    
    T = {
        "person": f"{base}, 1boy, solo, Chinese historical figure {name}, {dynasty} dynasty, {tag_str}, masculine man, short beard, stern face, traditional Chinese clothing, hanfu, ancient Chinese art style, ink painting, 2d illustration",
        "event": f"{base}, Chinese historical painting of {name}, {dynasty}, {tag_str}, ancient Chinese soldiers, dramatic battle scene, traditional Chinese painting, 2d illustration",
        "place": f"{base}, scenery, Chinese ancient landmark {name}, {dynasty}, {tag_str}, grand architecture, misty mountains, no humans, traditional Chinese landscape painting, 2d illustration",
        "weapon": f"{base}, Chinese ancient weapon {name}, {dynasty}, {tag_str}, ornate metalwork, still life, no humans, displayed on silk, traditional art style, 2d illustration",
        "classic": f"{base}, Chinese ancient classic {name}, {dynasty}, {tag_str}, bamboo slips, ink brush, scrolls, warm candlelight, no humans, still life, 2d illustration",
        "book": f"{base}, Chinese ancient classic {name}, {dynasty}, {tag_str}, bamboo slips, ink brush, scrolls, warm candlelight, no humans, still life, 2d illustration",
        "dynasty": f"{base}, Chinese imperial symbol {name}, {dynasty}, {tag_str}, dragon, jade seal, palace, Great Wall, majestic, 2d illustration",
    }
    return T.get(ctype, T["person"])

NEG = "1girl, female, feminine, woman, girl, lady, 3d, realistic, photo, photograph, western, european, modern, gun, text, watermark, signature, lowres, worst quality, ugly, deformed, blurry"
print("✅ 提示词就绪")

In [ ]:
# @title 6. 🚀 生成10张！
OUT = DRIVE / "card_test_anything"
OUT.mkdir(exist_ok=True)

for i, card in enumerate(cards):
    cid = card["card_id"]
    name = card["name"]
    ctype = card["type"]
    
    out_path = OUT / f"{cid}.png"
    if out_path.exists():
        print(f"[{i+1}/10] {name} (skip)")
        continue
    
    print(f"\n[{i+1}/10] {cid}: {name} ({ctype})")
    
    try:
        # 生成简单的构图参考（不需要网络下载）
        ref = Image.new("RGB", (832, 1216), (50, 45, 40))
        from PIL import ImageDraw
        draw = ImageDraw.Draw(ref)
        if ctype == "person":
            draw.ellipse((340, 150, 492, 310), outline=(120,100,80), width=4)
            draw.rectangle((312, 310, 520, 750), outline=(120,100,80), width=4)
        elif ctype == "place":
            draw.rectangle((100, 400, 350, 900), outline=(120,100,80), width=4)
            draw.polygon([(80,400),(225,180),(370,400)], outline=(120,100,80), width=4)
        elif ctype == "weapon":
            draw.line((416, 100, 416, 950), fill=(120,100,80), width=6)
            draw.rectangle((346, 900, 486, 960), outline=(120,100,80), width=3)
        
        edge = canny(ref, low_threshold=60, high_threshold=160)
        prompt = make_prompt(card)
        
        result = pipe(
            prompt=prompt, negative_prompt=NEG,
            image=edge, controlnet_conditioning_scale=0.8,
            num_inference_steps=28, guidance_scale=6.5,
            width=832, height=1216,
        ).images[0]
        
        result.save(out_path)
        print(f"  ✅ -> {cid}.png")
        gc.collect(); torch.cuda.empty_cache()
        
    except Exception as e:
        print(f"  ❌ {str(e)[:80]}")

done = len(list(OUT.glob("*.png")))
print(f"\n🎉 {done}/10 张完成! 保存在: Google Drive/card_test_anything/")